# Day 13 - Softmax Reduction Benchmark

**Original extension:** matched comparison with
[`../../cute-dsl/day11_softmax_reduction/`](../../cute-dsl/day11_softmax_reduction/).

Day 02 follows Triton's official fused-softmax tutorial. This new day revisits
the operation as a cross-framework reduction lab: it adds adversarial
correctness tests, a reusable output-buffer launcher, matched shapes, and
benchmarks against PyTorch and CuTe DSL.

## Challenge

Implement stable row-wise softmax for a contiguous FP32 matrix:

```text
row_max = max(x)
numerator = exp(x - row_max)
output = numerator / sum(numerator)
```

1. Assign one Triton program to each row.
2. Round `n_cols` up to a power of two for `tl.arange`.
3. Mask padded columns and load them as negative infinity.
4. Use `tl.max` and `tl.sum` along the program's vector axis.
5. Store only valid columns.
6. Choose `num_warps` from the padded block size.

`launch_softmax(x, y)` writes into a preallocated output so the benchmark does
not repeatedly allocate the Triton output.

## Tests

`test_softmax()` checks:

- `1x1` and rows shorter than a warp;
- 128- and 512-column regular rows;
- irregular width 1003;
- large positive and negative logits;
- elementwise agreement with `torch.softmax`;
- every output row sums to one.

The max subtraction is observable in the extreme-value case. An unstable
implementation will overflow or produce NaNs.

## Benchmark

```bash
python day13_softmax_reduction/solution.py --benchmark
```

The benchmark uses the same FP32 column counts as CuTe Day 11: 128, 512, 1003,
and 1024. It reports Triton and PyTorch time plus effective bandwidth:

```text
GB/s = 2 * rows * columns * sizeof(float) / elapsed_seconds / 1e9
```

The two represents one input read and one output write. This is a useful
traffic-normalized metric, but softmax also performs exponentials and two
reductions, so it is not a pure copy-bandwidth measurement.

## Compare with CuTe

| Concern | Triton | CuTe DSL |
| --- | --- | --- |
| Work mapping | One program owns a logical row vector | One explicit warp owns a row |
| Local values | `tl.arange` vector | Per-lane register tensor |
| Reduction | `tl.max`, `tl.sum` | `TensorSSA.reduce` plus warp shuffles |
| Tail handling | Masked load/store | `-inf` register initialization plus guarded accesses |
| Parallelism | Compiler lowers program reduction | Programmer specifies lane communication |
| Tuning knob | `BLOCK_SIZE`, `num_warps` | Values per lane, warps per row |

Triton's source is shorter because the compiler owns the cross-lane reduction
lowering. CuTe exposes ownership and communication directly, making it easier
to teach exactly where the reduction occurs and to replace the one-warp design
with a custom multi-warp shared-memory tree.

Use the CuTe Day 11 notebook for a side-by-side benchmark under one timing
harness. Compare measured output rather than assuming either abstraction is
faster: row width, register pressure, compiler version, and GPU architecture
all affect the result.

## Notebook workflow

Original extension: compare Triton program-level reductions with CuTe Day 11. Complete the stable softmax, run the fixed tests, then benchmark the matched shapes.

In [ ]:
#!/usr/bin/env python3
"""Day 13 puzzle: stable row softmax with tests and benchmarks."""

from __future__ import annotations

import argparse

import torch
import triton
import triton.language as tl


@triton.jit
def softmax_kernel(x_ptr, y_ptr, n_cols, BLOCK_SIZE: tl.constexpr):
    # EXERCISE(1): load one padded row, subtract its maximum, exponentiate,
    # reduce the denominator, and store only valid columns.
    pass


def _validate_input(x: torch.Tensor) -> None:
    if not x.is_cuda:
        raise ValueError("x must be a CUDA tensor")
    if x.dtype != torch.float32:
        raise TypeError("x must have dtype torch.float32")
    if x.ndim != 2 or not x.is_contiguous():
        raise ValueError("x must be a contiguous 2-D tensor")
    if x.shape[0] == 0 or x.shape[1] == 0:
        raise ValueError("x dimensions must be non-zero")
    if x.shape[1] > 65_536:
        raise ValueError("row is too wide for this one-program lesson")


def launch_softmax(x: torch.Tensor, y: torch.Tensor) -> None:
    # EXERCISE(2): choose the power-of-two block size, tune num_warps, and
    # launch one Triton program per row.
    raise NotImplementedError("Day 13 EXERCISE(2): launch softmax")


def softmax(x: torch.Tensor) -> torch.Tensor:
    y = torch.empty_like(x)
    launch_softmax(x, y)
    return y


def test_softmax() -> None:
    torch.manual_seed(20260915)
    for shape in ((1, 1), (7, 31), (33, 128), (65, 512), (257, 1003)):
        x = torch.randn(shape, device="cuda", dtype=torch.float32) * 7.0
        actual = softmax(x)
        expected = torch.softmax(x, dim=1)
        torch.testing.assert_close(actual, expected, atol=2e-6, rtol=2e-5)
        torch.testing.assert_close(
            actual.sum(dim=1),
            torch.ones(shape[0], device="cuda"),
            atol=2e-6,
            rtol=2e-6,
        )

    extreme = torch.tensor(
        [[-1000.0, 0.0, 1000.0], [1000.0, 1000.0, 999.0]],
        device="cuda",
    )
    torch.testing.assert_close(
        softmax(extreme),
        torch.softmax(extreme, dim=1),
        atol=2e-6,
        rtol=2e-5,
    )
    print("Day 13 Triton softmax: correctness OK")


def benchmark_softmax(
    rows: int,
    columns: int,
    warmup: int = 25,
    repetitions: int = 100,
) -> tuple[float, float, float]:
    x = torch.randn(rows, columns, device="cuda", dtype=torch.float32)
    y = torch.empty_like(x)
    launch_softmax(x, y)
    torch.cuda.synchronize()
    triton_ms = triton.testing.do_bench(
        lambda: launch_softmax(x, y),
        warmup=warmup,
        rep=repetitions,
    )
    torch_ms = triton.testing.do_bench(
        lambda: torch.softmax(x, dim=1),
        warmup=warmup,
        rep=repetitions,
    )
    bytes_moved = 2 * x.numel() * x.element_size()
    return triton_ms * 1_000, bytes_moved / (triton_ms * 1e6), torch_ms * 1_000


def benchmark_suite(rows: int = 4096) -> None:
    print("columns | Triton us | Torch us | effective GB/s")
    print("------- | --------- | -------- | --------------")
    for columns in (128, 512, 1003, 1024):
        triton_us, bandwidth, torch_us = benchmark_softmax(rows, columns)
        print(
            f"{columns:7d} | {triton_us:9.2f} | {torch_us:8.2f} | "
            f"{bandwidth:14.2f}"
        )


def run(rows: int, benchmark: bool) -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("Day 13 needs a CUDA device")
    test_softmax()
    if benchmark:
        benchmark_suite(rows)

## Correctness

Run fixed regular, irregular, and extreme-logit cases before measuring speed.

In [ ]:
test_softmax()

## Benchmark

Use a preallocated Triton output and report both latency and effective bandwidth.

In [ ]:
benchmark_suite(rows=4096)